# HighPitchCT (`v1.0.0`) — تدريب محطات العمل الشامل (`35` مريضاً، `~50,000` شريحة، `100 Epoch` مع `Val` و`--resume`) + دراسة الحذف الكاملة (`Stage 7 Ablation M0..M5`)

هذا الدفتر مُصمَّم لبيئات **`NVIDIA RTX PRO 6000 (96 GB VRAM)`** أو **`NVIDIA A100 (80 GB VRAM)`** مع **`~150 GB RAM`** ومساحة قرص **`>= 120 GB`**.
- يدعم **الاستئناف التلقائي الكامل (`--resume`)**: إذا توقفت الجلسة أثناء التنزيل أو أثناء بناء بيانات المرضى (`Helical3D / SSRB / FBP`) أو في منتصف الـ `100 Epoch`، أعد تشغيل نفس الخلية فقط وسيكمل فوراً من حيث توقف بالضبط.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# عدّل WORKSPACE_ROOT حسب مسار القرص السريع لديك في السيرفر (مثلاً /workspace أو /raid أو /data)
WORKSPACE_ROOT = Path(os.environ.get('HPCT_WORKSPACE', '/workspace'))
REPO_DIR = WORKSPACE_ROOT / 'HighPitchCT-Pipeline'
DATA_ROOT = WORKSPACE_ROOT / 'highpitchct-data'
MANIFEST_CSV = DATA_ROOT / 'manifests' / 'cohort_35_patients.csv'

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print('=== [1/4] Cloning / Updating HighPitchCT-Pipeline (v1.0.0) ===', flush=True)
if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git', str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', 'origin/main'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

import torch
import highpitchct
print(f'Installed highpitchct v{highpitchct.__version__} | CUDA={torch.cuda.is_available()} | GPUs={torch.cuda.device_count()}', flush=True)
for gi in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(gi)
    print(f'  -> cuda:{gi}: {p.name} ({p.total_memory / (1024**3):.1f} GiB VRAM)', flush=True)


In [ ]:
print('\n=== [2/4] Discovering & Downloading 35 Patients (CHEST + ABDOMEN) from TCIA ===', flush=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src')

if not MANIFEST_CSV.is_file():
    subprocess.run([
        sys.executable, '-m', 'highpitchct.cli', 'discover-tcia',
        '--body-part', 'CHEST,ABDOMEN',
        '--roles', 'projection_full,reconstruction_published',
        '--max-cases', '35',
        '--output', str(MANIFEST_CSV),
    ], env=env, check=True)

subprocess.run([
    sys.executable, '-m', 'highpitchct.cli', 'download',
    '--series-file', str(MANIFEST_CSV),
    '--data-root', str(DATA_ROOT),
    '--profile', 'full',
    '--cleanup-zip',
    '--workers', '8',
], env=env, check=True)


In [ ]:
print('\n=== [3/4] Full-Volume Sharding (~50,000 Slices) + 100-Epoch Val Training + Stage 7 Ablation ===', flush=True)
cmd_scale = [
    sys.executable, '-m', 'highpitchct.cli', 'stage6-scale',
    '--data-root', str(DATA_ROOT),
    '--n-train', '25',
    '--n-val', '5',
    '--n-test', '5',
    '--target-total-samples', '50000',
    '--pitches', '1.5,2.0,2.5',
    '--scenario', 'both',
    '--itf-ratio', '0.60',
    '--image-size', '256',
    '--epochs', '100',
    '--batch-size', '24',
    '--eval-batch-size', '32',
    '--lr', '1.5e-3',
    '--sino-channels', '64',
    '--img-channels', '64',
    '--img-in-channels', '4',
    '--backend', 'torch_cuda',
    '--amp', 'bf16',
    '--workers', '12',
    '--num-loader-workers', '6',
    '--run-stage7-ablation',
    '--m3-epochs', '15',
    '--resume',
]

# نمرر stdout/stderr مباشرةً لتظهر نسب التقدم % و ETA ومقاييس Val لكل Epoch لحظةً بلحظة
subprocess.run(cmd_scale, env=env, check=True)


In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

print('=== [4/4] عرض تقارير التحقق والاختبار ومصفوفة الحذف (Stage 7 Ablation M0..M5) ===')
scale_report_path = DATA_ROOT / 'manifests' / 'stage6_scale_report.json'
ablation_csv_path = DATA_ROOT / 'manifests' / 'stage7_ablation' / 'stage7_ablation_table.csv'
qa_grid_png = DATA_ROOT / 'manifests' / 'figures' / 'stage7_ablation_visual_qa.png'
curves_png = DATA_ROOT / 'manifests' / 'figures' / 'stage7_pitch_degradation_curves.png'

if scale_report_path.is_file():
    rep = json.loads(scale_report_path.read_text(encoding='utf-8'))
    print('أفضل Epoch على بيانات التحقق (Val):', rep['best_val_epoch'])
    print('تدقيق عدم تسرب البيانات (25/5/5):', json.dumps(rep['zero_leakage_audit'], indent=2, ensure_ascii=False))
    print('نتائج مرضى الاختبار الخمسة غير المرئيين (Unseen Test):', json.dumps(rep['unseen_test_metrics'], indent=2, ensure_ascii=False))

if ablation_csv_path.is_file():
    display(Markdown('### جدول مقارنة الطرق الست (`M0` إلى `M5`) عبر `ITF` و`RDC` وجميع قيم الـ Pitch'))
    display(pd.read_csv(ablation_csv_path))

if qa_grid_png.is_file():
    display(Markdown('### لوحة المقارنة البصرية الثمانية (`Stage 7 Visual QA`)'))
    display(Image(filename=str(qa_grid_png)))

if curves_png.is_file():
    display(Markdown('### منحنيات تدهور `RMSE` و`SSIM` و`PSNR` بدلالة الـ Pitch (`M0..M5`)'))
    display(Image(filename=str(curves_png)))
